# Structured streaming Demo

In [30]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime




In [31]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .getOrCreate()


In [32]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [33]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [34]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [35]:

# Use console output (goes to Docker logs)
streaming_query_df = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [36]:
streaming_query_df.stop()

In [37]:
from pyspark.sql.functions import avg, sum as _sum, max as _max, abs as _abs, desc

enriched_df = parsed_df \
    .withColumn("event_time", col("timestamp").cast("timestamp")) \
    .withColumn("edit_size", col("length.new") - col("length.old"))

In [38]:
size_by_bot_df = enriched_df \
    .groupBy(window(col("event_time"), "10 minutes"), col("bot")) \
    .agg(
        count("*").alias("nb_edits"),
        avg("edit_size").alias("taille_moyenne"),
        _sum(_abs(col("edit_size"))).alias("octets_modifies"),
        _max(_abs(col("edit_size"))).alias("plus_grosse_modif")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "bot", "nb_edits", "taille_moyenne", "octets_modifies", "plus_grosse_modif"
    )

query_size = size_by_bot_df \
    .writeStream \
    .queryName("taille_par_bot") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [39]:
top_users_df = enriched_df \
    .groupBy(col("user"), col("bot")) \
    .agg(
        count("*").alias("nb_edits"),
        _sum(col("edit_size")).alias("octets_ajoutes")
    ) \
    .orderBy(desc("nb_edits")) \
    .limit(10)

query_users = top_users_df \
    .writeStream \
    .queryName("top_users") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [40]:
top_pages_df = enriched_df \
    .groupBy(col("title")) \
    .agg(
        count("*").alias("nb_edits"),
        _sum(_abs(col("edit_size"))).alias("octets_modifies")
    ) \
    .orderBy(desc("nb_edits")) \
    .limit(10)

query_pages = top_pages_df \
    .writeStream \
    .queryName("top_pages") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [41]:
minor_df = enriched_df \
    .groupBy(window(col("event_time"), "30 minutes"), col("minor")) \
    .agg(
        count("*").alias("nb_edits"),
        avg(_abs(col("edit_size"))).alias("taille_moyenne_abs")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "minor", "nb_edits", "taille_moyenne_abs"
    )

query_minor = minor_df \
    .writeStream \
    .queryName("mineures_vs_majeures") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [49]:
for q in spark.streams.active:
    q.stop()

In [50]:
raw_filtered_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams_filtered") \
    .option("startingOffsets", "earliest") \
    .load()

filtered_df = raw_filtered_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*") \
 .withColumn("event_time", col("timestamp").cast("timestamp")) \
 .withColumn("edit_size", col("length.new") - col("length.old"))

In [44]:
check_df = filtered_df.groupBy("bot").count()

query_check = check_df \
    .writeStream \
    .queryName("verif_filtre_bots") \
    .format("console") \
    .outputMode("complete") \
    .start()

In [51]:
for q in spark.streams.active:
    q.stop()

spark.conf.set("spark.sql.shuffle.partitions", 4)

In [52]:
tumbling_df = enriched_df \
    .groupBy(window(col("event_time"), "10 minutes"), col("bot")) \
    .agg(
        count("*").alias("nb_edits"),
        avg(_abs(col("edit_size"))).alias("taille_moyenne_abs")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "bot", "nb_edits", "taille_moyenne_abs"
    ) \
    .orderBy("window_start", "bot")

query_tumbling = tumbling_df \
    .writeStream \
    .queryName("tumbling_10min") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .option("numRows", 100) \
    .start()

In [53]:
query_tumbling.stop()

sliding_df = enriched_df \
    .groupBy(window(col("event_time"), "10 minutes", "5 minutes"), col("bot")) \
    .agg(
        count("*").alias("nb_edits"),
        avg(_abs(col("edit_size"))).alias("taille_moyenne_abs")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "bot", "nb_edits", "taille_moyenne_abs"
    ) \
    .orderBy("window_start", "bot")

query_sliding = sliding_df \
    .writeStream \
    .queryName("sliding_10min_pas_5min") \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .option("numRows", 100) \
    .start()

In [54]:
from pyspark.sql.functions import sum as _sum

query_sliding.stop()

batch_df = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load() \
    .select(from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")) \
    .select("data.*") \
    .withColumn("event_time", col("timestamp").cast("timestamp"))

total_events = batch_df.count()

total_tumbling = batch_df.groupBy(window(col("event_time"), "10 minutes")) \
    .count().agg(_sum("count")).collect()[0][0]

total_sliding = batch_df.groupBy(window(col("event_time"), "10 minutes", "5 minutes")) \
    .count().agg(_sum("count")).collect()[0][0]

print(f"Nombre d'événements           : {total_events}")
print(f"Somme des comptes (tumbling)  : {total_tumbling}")
print(f"Somme des comptes (glissante) : {total_sliding}")
print(f"Rapport glissante / tumbling  : {total_sliding / total_tumbling:.2f}")

Nombre d'événements           : 1750
Somme des comptes (tumbling)  : 1750
Somme des comptes (glissante) : 3500
Rapport glissante / tumbling  : 2.00
